<a href="https://colab.research.google.com/github/irurama2018-coder/M640-Transformer-AI-Capstone/blob/main/QM640_RQ3_RQ4_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM640 — RQ3 and RQ4 Python Analysis

**RQ3:** Distribution-transformer benchmark loss comparisons and loading sensitivity. **RQ4:** NPV, hypothetical incremental premiums, Monte Carlo, and exact binary portfolio selection.

**Evidence limits:** Loss values are published maximum-loss benchmarks, not factory tests. Loading and all financial inputs are assumptions. Power transformers require separate verified inputs.

Place the supplied input CSV alongside this notebook. In Google Colab, upload the CSV when prompted.

In [1]:
# Google Colab users: uncomment the next two lines to upload the CSV
# from google.colab import files
# files.upload()

In [3]:
"""QM640 RQ3/RQ4: reproducible distribution-transformer engineering scenarios.
Run: python QM640_RQ3_RQ4_Analysis.py
Requires: numpy, matplotlib. Input CSV must be beside this script.
Not measured utility savings; BEE benchmark ceiling comparisons and hypothetical finance.
"""
from pathlib import Path
import csv
import itertools
import numpy as np
import matplotlib.pyplot as plt

BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
INPUT = BASE / 'QM640_RQ34_Distribution_Benchmark_Inputs.csv'
OUT = BASE / 'QM640_RQ34_Python_Results'
OUT.mkdir(exist_ok=True)
LOAD = 0.60
HOURS = 8760
ENERGY_INR_KWH = 7.0
DISCOUNT_RATE = 0.08
YEARS = 15
BUDGET_INR = 800000
SEED = 6402026
DRAWS = 5000

def read_data(path):
    with path.open(newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

def infer_losses(p50_w, p100_w):
    """Infer kW components from total-loss ceilings, NOT OEM measured components."""
    p0_w = (4 * p50_w - p100_w) / 3
    pll_w = 4 * (p100_w - p50_w) / 3
    if p0_w < 0 or pll_w < 0:
        raise ValueError('Inconsistent total-loss benchmark inputs')
    return p0_w / 1000, pll_w / 1000

def annual_loss_kwh(p0_kw, pll_kw, load_fraction, hours=HOURS):
    return (p0_kw + load_fraction**2 * pll_kw) * hours

def annuity_factor(rate=DISCOUNT_RATE, years=YEARS):
    return (1 - (1 + rate)**(-years)) / rate if rate else years

def write_csv(path, header, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)

records=[]
for row in read_data(INPUT):
    rating=int(row['rating_kva'])
    p03,pll3=infer_losses(float(row['p50_3star_w']),float(row['p100_3star_w']))
    p05,pll5=infer_losses(float(row['p50_5star_w']),float(row['p100_5star_w']))
    capex=float(row['full_replacement_capex_inr'])
    saving=annual_loss_kwh(p03,pll3,LOAD)-annual_loss_kwh(p05,pll5,LOAD)
    annual_value=saving*ENERGY_INR_KWH
    npv=-capex+annual_value*annuity_factor()
    records.append(dict(rating=rating,p03=p03,pll3=pll3,p05=p05,pll5=pll5,capex=capex,
                        savings_kwh=saving,annual_value=annual_value,npv=npv))

# RQ3: loss and loading sensitivity
write_csv(OUT/'RQ3_Base_Results.csv',
          ['rating_kva','p0_3star_kw','pll_3star_kw','p0_5star_kw','pll_5star_kw','annual_savings_kwh','annual_value_inr'],
          [[r['rating'],r['p03'],r['pll3'],r['p05'],r['pll5'],r['savings_kwh'],r['annual_value']] for r in records])
sensitivity=[]
for r in records:
    for loading in [.25,.50,.60,.75,1.0]:
        b=annual_loss_kwh(r['p03'],r['pll3'],loading)
        e=annual_loss_kwh(r['p05'],r['pll5'],loading)
        sensitivity.append([r['rating'],loading,b,e,b-e,(b-e)*ENERGY_INR_KWH])
write_csv(OUT/'RQ3_Loading_Sensitivity.csv',
          ['rating_kva','loading_fraction','baseline_kwh','comparison_kwh','savings_kwh','annual_value_inr'],sensitivity)

# RQ4: full replacement versus incremental upgrade economics
# Incremental upgrade premiums below are HYPOTHETICAL, not public procurement observations.
PREMIUM_FRACTIONS=[.10,.20,.30]
finance=[]
for r in records:
    finance.append([r['rating'],'full_replacement',r['capex'],r['npv']])
    for fraction in PREMIUM_FRACTIONS:
        cost=r['capex']*fraction
        finance.append([r['rating'],f'incremental_premium_{int(fraction*100)}pct',cost,
                        -cost+r['annual_value']*annuity_factor()])
write_csv(OUT/'RQ4_NPV_Scenarios.csv',['rating_kva','scenario','invested_capex_inr','npv_inr'],finance)

# Exact binary portfolio optimization by enumerating 2^6=64 feasible/infeasible portfolios.
# This solves the same binary budget-selection model as MILP for this six-option case.
def optimize(costs, values, budget):
    best_value=0.;best_cost=0.;best_bits=(0,)*len(costs)
    for bits in itertools.product((0,1),repeat=len(costs)):
        cost=sum(c*b for c,b in zip(costs,bits))
        value=sum(v*b for v,b in zip(values,bits))
        if cost<=budget+1e-9 and value>best_value+1e-9:
            best_value,best_cost,best_bits=value,cost,bits
    return best_value,best_cost,best_bits

costs=[r['capex'] for r in records]; values=[r['npv'] for r in records]
portfolio=[]
for budget in [200000,400000,600000,800000,1000000]:
    objective,spent,bits=optimize(costs,values,budget)
    portfolio.append([budget,spent,objective,','.join(str(r['rating']) for r,b in zip(records,bits) if b) or 'None'])
write_csv(OUT/'RQ4_Exact_Portfolio_Optimization.csv',
          ['budget_inr','selected_capex_inr','portfolio_npv_inr','selected_ratings_kva'],portfolio)

# RQ4 Monte Carlo: independent triangular assumptions, 5,000 draws per rating.
# Fixed seed; outcomes are scenario frequencies, not empirical probabilities.
rng=np.random.default_rng(SEED)
mc=[];mc_draws=[]
for r in records:
    loads=rng.triangular(.35,.60,.85,DRAWS)
    prices=rng.triangular(5,7,10,DRAWS)
    cost_factors=rng.triangular(.8,1,1.2,DRAWS)
    delta_kw=(r['p03']-r['p05'])+loads**2*(r['pll3']-r['pll5'])
    npvs=-r['capex']*cost_factors+delta_kw*HOURS*prices*annuity_factor()
    mc.append([r['rating'],DRAWS,float(np.mean(npvs)),float(np.quantile(npvs,.05)),
               float(np.quantile(npvs,.95)),float(np.mean(npvs>0))])
    mc_draws.extend([[r['rating'],i+1,float(loads[i]),float(prices[i]),float(cost_factors[i]),float(npvs[i])]
                     for i in range(DRAWS)])
write_csv(OUT/'RQ4_Monte_Carlo_Summary.csv',
          ['rating_kva','draws','mean_npv_inr','p05_npv_inr','p95_npv_inr','positive_npv_fraction'],mc)
write_csv(OUT/'RQ4_Monte_Carlo_Draws.csv',
          ['rating_kva','draw','loading_fraction','energy_inr_per_kwh','capex_multiplier','npv_inr'],mc_draws)

# Charts
fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar([str(r['rating']) for r in records],[r['savings_kwh'] for r in records]);ax.set(xlabel='Transformer rating (kVA)',ylabel='Modeled annual energy savings (kWh)',title='RQ3: Benchmark-ceiling comparison at 60% loading')
fig.tight_layout();fig.savefig(OUT/'RQ3_Annual_Savings.png',dpi=170);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4.5))
for r in records:
    xs=np.array([.25,.5,.6,.75,1.]);ys=[annual_loss_kwh(r['p03'],r['pll3'],x)-annual_loss_kwh(r['p05'],r['pll5'],x) for x in xs]
    ax.plot(xs*100,ys,marker='o',label=f"{r['rating']} kVA")
ax.set(xlabel='Loading (%)',ylabel='Modeled annual savings (kWh)',title='RQ3: Loading sensitivity');ax.legend(ncol=3,fontsize=8)
fig.tight_layout();fig.savefig(OUT/'RQ3_Loading_Sensitivity.png',dpi=170);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar([str(r['rating']) for r in records],[r['npv'] for r in records]);ax.axhline(0,linewidth=1)
ax.set(xlabel='Transformer rating (kVA)',ylabel='NPV (INR)',title='RQ4: Hypothetical full-replacement energy-only NPV')
fig.tight_layout();fig.savefig(OUT/'RQ4_NPV.png',dpi=170);plt.close(fig)

print('Completed:',OUT)
print('RQ3 annual savings (kWh):',[(r['rating'],round(r['savings_kwh'],2)) for r in records])
print('RQ4 optimal portfolio at INR 800,000:',portfolio[3])
print('Monte Carlo positive-NPV fractions:',[(row[0],row[-1]) for row in mc])
print('Note: power-transformer calculations intentionally excluded until comparable verified inputs are available.')


FileNotFoundError: [Errno 2] No such file or directory: '/content/QM640_RQ34_Distribution_Benchmark_Inputs.csv'